# <div align="center"> Statistical Learning </div>
# <div align="center"> Machine Learning and Econometrics </div>
## <div align="center"> ECO 4971/6971 </div>
#### <div align="center">Class 9 — Resampling Methods: Exercise Solutions</div>
<div align="center"> Jonathan Holmes (he/him)</div>

These exercises use the `Auto.csv` dataset and concern polynomial regression of `mpg` on `horsepower`.

# Exercise 1 — $p$-values

**Q1. Using just the estimates and $p$-values, what level of polynomial is best?**

---

**Answer.**

We fit polynomial regressions of increasing degree and examine whether the highest-order term is statistically significant. The procedure is:

* Start with a linear model ($p=1$). If $\hat{\beta}_1$ is significant, try degree 2.
* Add degree 2. If $\hat{\beta}_2$ is significant, try degree 3.
* Continue until the new term's $p$-value exceeds 0.05.

In this dataset the quadratic term ($p=2$) is strongly significant and the cubic term ($p=3$) is generally not, suggesting **degree 2** (a quadratic) is the appropriate choice by this criterion.

> **Limitation:** sequential hypothesis testing inflates the Type I error rate and does not account for the full model complexity. The next exercises use proper model-selection criteria.

# Exercise 2 — Corrected In-Sample Statistics

**Q2. According to the corrected in-sample statistics (AIC, BIC, Adjusted $R^2$), how many polynomial terms should we include?**

---

**Answer.**

All three criteria penalise model complexity to correct for the fact that adding parameters always reduces training RSS:

* **AIC** $= n\log(\widehat{\sigma}^2) + 2p$ — penalty is $2p$.
* **BIC** $= n\log(\widehat{\sigma}^2) + p\log(n)$ — penalty is $p\log(n)$, which is heavier than AIC for $n > 8$.
* **Adjusted $R^2$** $= 1 - \dfrac{RSS/(n-p-1)}{TSS/(n-1)}$ — penalises each additional predictor by losing a degree of freedom.

We minimise AIC and BIC; we maximise Adjusted $R^2$. In the `Auto` demonstration, these criteria generally favour **degree 2**, with some criteria occasionally pointing to degree 3. The corrected statistics confirm the $p$-value result.

---

**Q3. Does each method select a different number of polynomial terms?**

---

**Answer.**

Not always — when the signal is clear, AIC, BIC, and Adjusted $R^2$ often agree. However:

* **BIC** has a stronger penalty for complexity, so it tends to select **fewer** terms than AIC.
* **AIC** may select one or two more terms than BIC.
* **Adjusted $R^2$** has the weakest effective penalty and sometimes agrees with AIC.

In this example the differences are small (degree 2 vs 3), and all three methods agree that a **quadratic** fit is far better than a linear one.

# Advanced Exercise — AIC vs BIC

**Q4. With other datasets, do you think AIC, BIC, and Adjusted $R^2$ will always select the same number of parameters?**

---

**Answer.**

No. The criteria embody different **theoretical objectives**:

* **AIC** is derived from information theory and targets the model that best approximates the true data-generating process among the candidates. It is an asymptotically efficient selector — it tends to pick larger models to minimise prediction error.
* **BIC** is derived from Bayesian model selection and is **consistent**: as $n \to \infty$, BIC selects the true model if it is in the candidate set. Its heavier penalty ($\log n$ vs $2$) leads to systematically sparser models.
* **Adjusted $R^2$** is an ad hoc correction and does not have the same theoretical grounding.

In practice, disagreements are common when the "correct" model is ambiguous (e.g., weak signal, many similarly performing models). The choice of criterion should reflect the goal: **prediction** (favour AIC) or **inference/parsimony** (favour BIC).

---

**Q5. If you select $P$ using AIC (Model 1) and BIC (Model 2), which has higher bias? Which has higher variance?**

---

**Answer.**

Because BIC has a heavier complexity penalty, it typically selects a **simpler (smaller) model** than AIC:

| | AIC model | BIC model |
|---|---|---|
| Complexity | Higher (more terms) | Lower (fewer terms) |
| **Bias** | **Lower** | **Higher** |
| **Variance** | **Higher** | **Lower** |

Intuitively: BIC's sparser model underfits slightly more (higher bias) but is more stable across datasets (lower variance). AIC's more flexible model captures more signal but is also more sensitive to sampling variation (higher variance).

# Exercise 3 — Validation Set Approach

**Q6. Using the validation set approach, which polynomial degree is best?**

---

**Answer.**

In the validation set approach, the data are split (typically 50/50 or 70/30) into a training set and a held-out validation set. We fit each polynomial degree on the training set and compute **MSE on the validation set**. The best degree is the one with the **lowest validation MSE**.

In the class demonstration with `Auto`, the validation MSE curve is U-shaped, bottoming out at **degree 2**, consistent with the earlier criteria.

---

**Q7. If I re-ran this exercise, would I get the same answer?**

---

**Answer.**

Not necessarily. The validation set approach depends on the **random split** used to divide the data. A different random seed produces a different training set and validation set, which can shift the validation MSE curve and potentially change the selected degree by 1 or 2.

This **high variability** is the main weakness of the simple validation set approach — the model selection decision can change substantially depending on which observations happen to fall in the validation set. Cross-validation addresses this by averaging over many splits.

---

**Q8. What would happen to bias and variance if I changed the leave-out fraction from 0.5 to 0.2?**

---

**Answer.**

Changing the leave-out fraction from 0.5 to 0.2 means:
* **Training set grows** from 50% to 80% of the data.
* **Validation set shrinks** from 50% to 20%.

Effects on the **MSE estimate**:

| Effect | Direction | Reason |
|--------|-----------|--------|
| **Bias** | **Decreases** | The model is trained on more data, so the fitted model is closer to one trained on the full dataset. The MSE estimate better reflects full-data performance. |
| **Variance** | **Increases** | The validation set is smaller (20% of $n$), so the MSE estimate is based on fewer observations and is noisier. |

This is the same bias–variance tradeoff that governs the choice of $K$ in $K$-fold cross-validation.

# Exercise 4 — $K$-Fold Cross-Validation

**Q9. As you increase the number of folds $K$ in $K$-fold cross-validation, what happens to bias and variance?**

---

**Answer.**

| $K$ | Training set size | Validation set size | Bias of CV estimate | Variance of CV estimate |
|-----|------------------|--------------------|--------------------|------------------------|
| Small (e.g., $K=2$) | 50% | 50% | High | Low |
| Medium ($K=5$ or $10$) | 80–90% | 10–20% | Moderate | Moderate |
| Large ($K=n$, LOOCV) | $\frac{n-1}{n}\approx 100\%$ | 1 observation | **Low** | **High** |

**As $K$ increases:**

* **Bias decreases**: each training fold contains more observations, so the model trained on each fold closely approximates a model trained on the full dataset. The CV estimate is therefore less downward-biased.
* **Variance increases**: at large $K$, the $K$ training sets overlap heavily (they differ by only 1–2 observations), so the $K$ individual MSE estimates are highly correlated. The average of highly correlated quantities has high variance.

In practice, **$K = 5$ or $K = 10$** is a widely used compromise, offering low bias with manageable variance and computational cost.